In [0]:
SELECT
  season,
  COUNT(*) - COUNT(attendance) AS bez_posete
FROM
  bootcamp.northern_star.matches_flat
GROUP BY
  season;

SELECT
  stadium_name,
  MAX(attendance) AS maksimalna_poseta,
  MIN(attendance) AS minimalna_poseta
FROM
  bootcamp.northern_star.matches_flat
GROUP BY
  stadium_name
ORDER BY
  maksimalna_poseta DESC;

SELECT
  season,
  SUM(home_goals) AS golovi_domacina,
  SUM(away_goals) AS golovi_gostiju
FROM
  bootcamp.northern_star.matches_flat
GROUP BY
  season;

SELECT
  date_format(kickoff_ts, 'EEEE') AS dani_u_nedelji,
  AVG(home_goals + away_goals) AS prosecan_br_golova,
  COUNT(*) AS br_utakmica
FROM
  bootcamp.northern_star.matches_flat
GROUP BY
  dani_u_nedelji;

--5
SELECT
  referee_name,
  SUM(home_red_cards + away_red_cards) AS ukupan_broj_crvenih
FROM
  bootcamp.northern_star.matches_flat
GROUP by
  referee_name
HAVING
  SUM(home_red_cards + away_red_cards) >= 5
ORDER BY
  ukupan_broj_crvenih DESC;

--6
SELECT
  home_team,
  season,
  AVG(home_goals) AS prosek_golova
FROM
  bootcamp.northern_star.matches_flat
WHERE
  season = '2024/25'
GROUP BY
  home_team,
  season
HAVING
  prosek_golova > 2;

--7
SELECT
  date_format(kickoff_ts, 'MMMM') AS mesec_u_god,
  SUM(home_goals + away_goals) AS ukupan_br_golova
FROM
  bootcamp.northern_star.matches_flat
GROUP BY
  mesec_u_god
ORDER BY
  ukupan_br_golova DESC
LIMIT 3;

-- 1.1. Prikažite klubove koji su u ulozi domaćina odigrali najmanje 10 utakmica. Za svaki klub prikažite broj utakmica, ukupan broj postignutih golova i prosečan broj golova po utakmici.
SELECT
  home_team,
  COUNT(*) AS broj_utakmica,
  SUM(home_goals) AS ukupan_br_golova,
  ROUND(AVG(home_goals), 2) AS prosek_golova
FROM
  bootcamp.northern_star.matches_flat
GROUP BY
  home_team
HAVING
  broj_utakmica >= 10;

-- 2. Za svaku sezonu prikažite broj utakmica u kojima je gost postigao najmanje tri gola, kao i ukupan broj golova koje su gosti postigli u tim utakmicama.
SELECT
  season,
  COUNT(*) AS broj_utakmica,
  SUM(away_goals) AS ukupan_br_golova
FROM
  bootcamp.northern_star.matches_flat
WHERE
  away_goals >= 3
GROUP BY
  season;

SELECT
  CASE
    WHEN hour(kickoff_ts) < 14 THEN 'Rano'
    WHEN hour(kickoff_ts) < 17 THEN 'Popodne'
    WHEN hour(kickoff_ts) >= 17 THEN 'Vece'
    ELSE NULL
  END AS kickoff_slot,
  COUNT(*) AS broj_utakmica
FROM
  bootcamp.northern_star.matches_flat
GROUP BY
  kickoff_slot;

-- 2. Разврстајте посету у категорије 'Без података' (NULL), 'Без публике' (0), 'Мала' (испод 20.000) и 'Велика', па избројте утакмице по категорији. Пазите на редослед WHEN услова.
SELECT
  CASE
    WHEN attendance IS NULL THEN 'Без података'
    WHEN
      attendance > 0
      AND attendance < 20000
    THEN
      'Мала'
    WHEN attendance >= 20000 THEN 'Велика'
    ELSE NULL
  END AS br_poseta,
  COUNT(*) AS kategorije
FROM
  bootcamp.northern_star.matches_flat
GROUP BY
  br_poseta;

-- 3. За сваког судију прикажите број утакмица, број утакмица са бар једним црвеним картоном и тај удео у процентима. Ко има највећи удео?
SELECT
  referee_name,
  COUNT(*) AS broj_utakmica,
  SUM(home_red_cards + away_red_cards) AS crveni_kartoni,
  SUM(
    CASE
      WHEN home_red_cards + away_red_cards >= 1 THEN 1
      ELSE 0
    END
  ) AS utakmice_sa_crvenim,
  SUM(
    CASE
      WHEN home_red_cards + away_red_cards >= 1 THEN 1
      ELSE 0
    END
  )
    / COUNT(*)
    * 100 AS procenat_udela
FROM
  bootcamp.northern_star.matches_flat
GROUP BY
  referee_name
ORDER BY
  crveni_kartoni DESC;

-- 1. За утакмице сезоне 2024/25 прикажите датум, име судије и његову националност. Колико редова очекујете? Која националност судија се најчешће појављује?
SELECT
  m.season,
  date(kickoff_ts) as date_of_game,
  r.referee_name,
  r.nationality
FROM
  matches AS m
    JOIN referees AS r
      ON m.referee_id = r.referee_id
WHERE
  season = '2024/25';

SELECT
  r.referee_name,
  r.nationality,
  COUNT(*) as broj_utakmica
FROM
  matches as m
    JOIN referees AS r
      ON m.referee_id = r.referee_id
WHERE
  m.season = '2024/25'
GROUP BY
  r.nationality,
  r.referee_name;

-- 2. Прикажите клубове и њихове стадионе са капацитетом. Колико редова враћа INNER JOIN, а колико LEFT JOIN? Који клуб има највећи стадион (order by)?

SELECT
  t.team_name,
  s.stadium_name,
  s.capacity
FROM
  teams as t
    LEFT JOIN stadiums AS s
      ON t.stadium_id = s.stadium_id;

-- 3. Колико голова је постигнуто главом (body_part = 'head') у свакој сезони (group by season)? Рачунајте и аутоголове.

SELECT me.body_part, SUM(m.home_goals + m.away_goals) as br_golova_glavom, m.season
FROM matches AS m
JOIN match_events AS me ON m.match_id = me.match_id
WHERE me.body_part = 'head'
GROUP BY m.season, me.body_part;

-- 4. Који играчи су добили највише жутих картона у сезони 2024/25 (рачунајте и други жути) (match_events)?

SELECT p.first_name, p.last_name, COUNT(*) AS zuti_kartoni
FROM matches AS m
JOIN match_events AS me ON m.match_id = me.match_id
JOIN players AS p ON me.player_id = p.player_id
WHERE season = '2024/25' AND me.event_type IN('yellow_card', 'second_yellow')
GROUP BY p.first_name, p.last_name
ORDER BY zuti_kartoni DESC;

-- 5. Колико играча нема ни један ред у player_match_stats, односно никад нису били у саставу за утакмицу?

SELECT pl.first_name, pl.last_name
FROM players AS pl
LEFT JOIN player_match_stats AS p ON pl.player_id = p.player_id
WHERE p.player_id IS NULL;

-- 6. Који судија је судио највише утакмица у којима је досуђен пенал (дат или промашен)?

SELECT r.referee_name, COUNT(*) AS broj_utakmica_sa_penalom
FROM referees AS r
JOIN matches AS m ON r.referee_id = m.referee_id
WHERE m.match_report ILIKE '%penalty%'
GROUP BY r.referee_name
ORDER BY broj_utakmica_sa_penalom DESC;

SELECT r.referee_name, COUNT (DISTINCT m.match_id) AS broj_utakmica_sa_penalom
FROM referees AS r
LEFT JOIN matches AS m ON r.referee_id = m.referee_id
JOIN match_events AS me ON me.match_id = m.match_id
WHERE me.event_type IN ('penalty_goal', 'penalty_miss')
GROUP BY r.referee_name
ORDER BY broj_utakmica_sa_penalom DESC;



-- 1. Ко је дао највише асистенција у сезони 2024/25? Пазите: асистент је у колони
-- related_player_id.

SELECT p.first_name, p.last_name, COUNT(*) AS broj_asistencija
FROM match_events AS me
JOIN matches AS m ON me.match_id = m.match_id
JOIN players AS p ON me.related_player_id = p.player_id
WHERE me.event_type = 'goal' AND related_player_id IS NOT NULL AND m.season = '2024/25'
GROUP BY p.first_name, p.last_name
ORDER BY broj_asistencija DESC;

-- 2. Колико голова су по сезонама дали нападачи (FW), везни (MF) и одбрамбени (DF) играчи?
SELECT m.season, COUNT(*) AS br_golova, p.position
FROM matches AS m
JOIN match_events AS me ON me.match_id = m.match_id
JOIN players AS p ON p.player_id = me.player_id
WHERE me.event_type IN('goal', 'penalty_goal') AND p.position IN ('FW', 'MF', 'DF')
GROUP BY m.season, p.position
ORDER BY m.season, p.position;

SELECT DISTINCT event_type 
FROM match_events;

-- 3. Колика је укупна посета по граду стадиона? Који град је имао највише гледалаца?
SELECT SUM(m.attendance) AS ukupna_poseta, s.city
FROM matches AS m
JOIN stadiums AS s ON m.stadium_id = s.stadium_id
GROUP BY s.city
ORDER BY ukupna_poseta DESC;
